# 🚗 QuantumDrive – Research Notebook

**Hybrid Classical–Quantum Perception System for Autonomous Vehicles**

This notebook demonstrates the core ML pipelines of QuantumDrive:

1. **Knowledge Distillation** – Train a lightweight student model from a ResNet-50 teacher
2. **Quantum ML** – Hybrid classical-quantum inference with VQC
3. **Synthetic Data Generation** – Domain-randomized driving datasets
4. **Experiment Tracking** – Logging and querying experiment results
5. **Model Inference** – Load checkpoints and run predictions

---

## 0 · Environment Setup

In [ ]:
# Install dependencies (Colab already has torch; add extras)
%pip install -q pennylane scikit-learn tensorboard Pillow

In [ ]:
import sys, os, json, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

print(f"Python  : {sys.version}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

---
## 1 · Knowledge Distillation

We train a compact **Student (MobileNet-style)** to mimic a large **Teacher (ResNet-50)** using:
- Soft-label KL-divergence loss
- Feature-map alignment (MSE between intermediate representations)
- Hard-label cross-entropy

For this demo we use synthetic random data; replace with real driving data in production.

In [ ]:
# ── Teacher Model (ResNet-50 backbone) ────────────────────────
import torchvision.models as models

class TeacherModel(nn.Module):
    def __init__(self, num_classes=5, pretrained=True):
        super().__init__()
        backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if pretrained else None)
        self.features = nn.Sequential(*list(backbone.children())[:-1])   # → (B, 2048, 1, 1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(2048, num_classes),
        )

    def forward(self, x):
        feat = self.features(x)
        return self.classifier(feat), feat.view(feat.size(0), -1)


# ── Student Model (lightweight) ──────────────────────────────
class StudentModel(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.adapt = nn.Linear(128, 2048)          # project to teacher dim
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        feat = self.features(x).view(x.size(0), -1)
        return self.classifier(feat), self.adapt(feat)

print("✓ Models defined")

In [ ]:
# ── Distillation Loss ────────────────────────────────────────
import torch.nn.functional as F

class DistillationLoss(nn.Module):
    """Combined KD + feature-alignment + hard-label loss."""
    def __init__(self, temperature=4.0, alpha=0.7, beta=0.15):
        super().__init__()
        self.T = temperature
        self.alpha = alpha      # weight for soft-label loss
        self.beta  = beta       # weight for feature-map loss
        self.ce    = nn.CrossEntropyLoss()
        self.mse   = nn.MSELoss()

    def forward(self, student_logits, teacher_logits, labels,
                student_features=None, teacher_features=None):
        # Soft-label KL divergence
        soft_s = F.log_softmax(student_logits / self.T, dim=1)
        soft_t = F.softmax(teacher_logits / self.T, dim=1)
        kd_loss = F.kl_div(soft_s, soft_t, reduction="batchmean") * (self.T ** 2)

        # Hard-label CE
        hard_loss = self.ce(student_logits, labels)

        loss = self.alpha * kd_loss + (1 - self.alpha) * hard_loss

        # Feature alignment
        if student_features is not None and teacher_features is not None:
            loss += self.beta * self.mse(student_features, teacher_features.detach())

        return loss

print("✓ Distillation loss ready")

In [ ]:
# ── Synthetic Data ────────────────────────────────────────────
NUM_CLASSES = 5
IMG_SIZE    = 64       # small for demo speed
N_TRAIN     = 256
N_VAL       = 64
BATCH       = 32

def make_dataset(n):
    X = torch.randn(n, 3, IMG_SIZE, IMG_SIZE)
    y = torch.randint(0, NUM_CLASSES, (n,))
    return TensorDataset(X, y)

train_loader = DataLoader(make_dataset(N_TRAIN), batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(make_dataset(N_VAL),   batch_size=BATCH)

print(f"✓ Datasets: {N_TRAIN} train, {N_VAL} val  |  {NUM_CLASSES} classes  |  {IMG_SIZE}×{IMG_SIZE}px")

In [ ]:
# ── Training Loop ─────────────────────────────────────────────
EPOCHS = 10
LR = 1e-3

teacher = TeacherModel(NUM_CLASSES, pretrained=True).to(DEVICE).eval()
student = StudentModel(NUM_CLASSES).to(DEVICE)

criterion = DistillationLoss(temperature=4.0, alpha=0.7, beta=0.15)
optimizer = optim.AdamW(student.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = {"epoch": [], "train_loss": [], "val_loss": [], "val_acc": []}

for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    student.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        with torch.no_grad():
            t_logits, t_feats = teacher(images)
        s_logits, s_feats = student(images)
        loss = criterion(s_logits, t_logits, labels, s_feats, t_feats)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
    scheduler.step()
    avg_train = running_loss / N_TRAIN

    # ── Validate ──
    student.eval()
    val_loss, correct = 0.0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            t_logits, t_feats = teacher(images)
            s_logits, s_feats = student(images)
            val_loss += criterion(s_logits, t_logits, labels, s_feats, t_feats).item() * images.size(0)
            correct += (s_logits.argmax(1) == labels).sum().item()
    avg_val = val_loss / N_VAL
    val_acc = correct / N_VAL * 100

    history["epoch"].append(epoch)
    history["train_loss"].append(avg_train)
    history["val_loss"].append(avg_val)
    history["val_acc"].append(val_acc)
    print(f"Epoch {epoch:>2}/{EPOCHS}  train_loss={avg_train:.4f}  val_loss={avg_val:.4f}  val_acc={val_acc:.1f}%")

print("\n✓ Knowledge distillation training complete")

In [ ]:
# ── Visualize Training Curves ─────────────────────────────────
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor("#0B0F19")
for ax in (ax1, ax2):
    ax.set_facecolor("#121826")
    ax.tick_params(colors="#8A94A6")
    ax.spines[:].set_color("#2A3550")

ax1.plot(history["epoch"], history["train_loss"], "-o", color="#5D8CFF", label="Train Loss")
ax1.plot(history["epoch"], history["val_loss"],   "-o", color="#FF5252", label="Val Loss")
ax1.set_title("Distillation Loss", color="white", fontsize=14)
ax1.set_xlabel("Epoch", color="#8A94A6")
ax1.legend(facecolor="#1F2A44", edgecolor="#2A3550", labelcolor="white")

ax2.plot(history["epoch"], history["val_acc"], "-o", color="#00E676")
ax2.set_title("Validation Accuracy (%)", color="white", fontsize=14)
ax2.set_xlabel("Epoch", color="#8A94A6")
ax2.set_ylim(0, 100)

plt.tight_layout()
plt.show()

---
## 2 · Quantum ML – Hybrid Classical-Quantum Inference

We build a **Variational Quantum Circuit (VQC)** using PennyLane:
- Classical encoder compresses features to `n_qubits` dimensions
- Quantum layer: AngleEmbedding → StronglyEntanglingLayers → PauliZ measurements
- Classical head maps quantum outputs to class logits

Falls back to a numpy-based simulator if no quantum hardware is available.

In [ ]:
# ── Quantum Neural Network ───────────────────────────────────
try:
    import pennylane as qml
    HAS_PENNYLANE = True
    print(f"✓ PennyLane {qml.__version__} available")
except ImportError:
    HAS_PENNYLANE = False
    print("⚠ PennyLane not installed – using numpy VQC fallback")

N_QUBITS = 4
N_LAYERS = 2

if HAS_PENNYLANE:
    dev = qml.device("default.qubit", wires=N_QUBITS)

    @qml.qnode(dev, interface="torch", diff_method="backprop")
    def quantum_circuit(inputs, weights):
        qml.AngleEmbedding(inputs, wires=range(N_QUBITS))
        qml.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
        return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

    class QuantumLayer(nn.Module):
        def __init__(self):
            super().__init__()
            weight_shape = qml.StronglyEntanglingLayers.shape(n_layers=N_LAYERS, n_wires=N_QUBITS)
            self.weights = nn.Parameter(torch.randn(*weight_shape) * 0.1)

        def forward(self, x):
            results = []
            for sample in x:
                out = quantum_circuit(sample, self.weights)
                results.append(torch.stack(out))
            return torch.stack(results)
else:
    # Numpy fallback VQC simulator
    class QuantumLayer(nn.Module):
        def __init__(self):
            super().__init__()
            self.W = nn.Parameter(torch.randn(N_QUBITS, N_QUBITS) * 0.1)

        def forward(self, x):
            rotated = torch.tanh(x @ self.W)
            return torch.cos(rotated)    # simulate Pauli-Z expectation values


class HybridQuantumNetwork(nn.Module):
    """Classical encoder → Quantum VQC → Classical head."""
    def __init__(self, input_dim=128, num_classes=5):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, N_QUBITS), nn.Tanh(),  # bound to [-1, 1] for angle embedding
        )
        self.quantum = QuantumLayer()
        self.head = nn.Sequential(
            nn.Linear(N_QUBITS, 32), nn.ReLU(),
            nn.Linear(32, num_classes),
        )

    def forward(self, x):
        z = self.encoder(x)
        q = self.quantum(z)
        return self.head(q)

print("✓ Hybrid quantum network defined")

In [ ]:
# ── Train Hybrid Quantum Model ────────────────────────────────
INPUT_DIM   = 128
QN_TRAIN    = 200
QN_EPOCHS   = 8

# Synthetic feature vectors
X_q = torch.randn(QN_TRAIN, INPUT_DIM)
y_q = torch.randint(0, NUM_CLASSES, (QN_TRAIN,))
q_loader = DataLoader(TensorDataset(X_q, y_q), batch_size=16, shuffle=True)

hybrid_model = HybridQuantumNetwork(INPUT_DIM, NUM_CLASSES).to(DEVICE)
q_optimizer  = optim.Adam(hybrid_model.parameters(), lr=5e-3)
q_criterion  = nn.CrossEntropyLoss()

q_history = []
for epoch in range(1, QN_EPOCHS + 1):
    hybrid_model.train()
    total_loss, total_correct = 0.0, 0
    for xb, yb in q_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = hybrid_model(xb)
        loss = q_criterion(logits, yb)
        q_optimizer.zero_grad()
        loss.backward()
        q_optimizer.step()
        total_loss += loss.item() * xb.size(0)
        total_correct += (logits.argmax(1) == yb).sum().item()
    acc = total_correct / QN_TRAIN * 100
    avg_loss = total_loss / QN_TRAIN
    q_history.append({"epoch": epoch, "loss": avg_loss, "acc": acc})
    print(f"Epoch {epoch}/{QN_EPOCHS}  loss={avg_loss:.4f}  acc={acc:.1f}%")

print("\n✓ Hybrid quantum training complete")

In [ ]:
# ── Quantum vs Classical Comparison ──────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor("#0B0F19")

for ax in (ax1, ax2):
    ax.set_facecolor("#121826")
    ax.tick_params(colors="#8A94A6")
    ax.spines[:].set_color("#2A3550")

epochs_q = [h["epoch"] for h in q_history]

ax1.plot(epochs_q, [h["loss"] for h in q_history], "-o", color="#A855F7", label="Quantum Hybrid")
ax1.plot(history["epoch"][:QN_EPOCHS], history["train_loss"][:QN_EPOCHS], "--s", color="#5D8CFF", label="Classical Student")
ax1.set_title("Loss Comparison", color="white", fontsize=14)
ax1.set_xlabel("Epoch", color="#8A94A6")
ax1.legend(facecolor="#1F2A44", edgecolor="#2A3550", labelcolor="white")

ax2.plot(epochs_q, [h["acc"] for h in q_history], "-o", color="#A855F7", label="Quantum Hybrid")
ax2.plot(history["epoch"][:QN_EPOCHS], history["val_acc"][:QN_EPOCHS], "--s", color="#5D8CFF", label="Classical Student")
ax2.set_title("Accuracy Comparison", color="white", fontsize=14)
ax2.set_xlabel("Epoch", color="#8A94A6")
ax2.set_ylim(0, 100)
ax2.legend(facecolor="#1F2A44", edgecolor="#2A3550", labelcolor="white")

plt.tight_layout()
plt.show()

---
## 3 · Synthetic Data Generation

Demonstrates the domain-randomization pipeline that generates diverse driving scenarios.

In [ ]:
# ── Domain Randomizer (standalone demo) ──────────────────────

WEATHER_PRESETS = [
    "ClearNoon", "CloudyNoon", "WetNoon", "HardRainNoon",
    "ClearSunset", "CloudySunset", "WetSunset", "HardRainSunset",
    "ClearNight", "CloudyNight", "WetNight", "HardRainNight",
]

VEHICLE_MODELS = [
    "vehicle.tesla.model3", "vehicle.audi.a2", "vehicle.bmw.grandtourer",
    "vehicle.chevrolet.impala", "vehicle.dodge.charger_police",
    "vehicle.mercedes.coupe", "vehicle.toyota.prius",
]

class DomainRandomizer:
    def __init__(self, seed=42):
        self.rng = random.Random(seed)

    def randomize_scene(self):
        return {
            "weather":         self.rng.choice(WEATHER_PRESETS),
            "num_vehicles":    self.rng.randint(10, 80),
            "num_pedestrians": self.rng.randint(5, 40),
            "ego_vehicle":     self.rng.choice(VEHICLE_MODELS),
            "sun_altitude":    round(self.rng.uniform(-30, 90), 1),
            "fog_density":     round(self.rng.uniform(0, 80), 1),
            "precipitation":   round(self.rng.uniform(0, 100), 1),
        }

randomizer = DomainRandomizer()
print("Generated 5 random driving scenes:\n")
for i in range(5):
    scene = randomizer.randomize_scene()
    print(f"  Scene {i+1}: {scene['weather']:18s} | {scene['num_vehicles']:2d} vehicles "
          f"| fog={scene['fog_density']:.0f}% | rain={scene['precipitation']:.0f}%")

In [ ]:
# ── Synthetic Frame Generator ─────────────────────────────────
# Simulates what CARLA would produce: an image tensor + bounding-box annotations

SIGN_CLASSES = ["stop", "speed_limit_30", "speed_limit_60", "yield", "no_entry", "traffic_light"]

def generate_synthetic_frame(img_size=(3, 256, 256)):
    """Generate a fake driving frame with random annotations."""
    image = torch.rand(*img_size)    # random noise image
    n_objects = random.randint(1, 8)
    annotations = []
    for _ in range(n_objects):
        x1, y1 = random.randint(0, 200), random.randint(0, 200)
        w, h = random.randint(20, 56), random.randint(20, 56)
        annotations.append({
            "class": random.choice(SIGN_CLASSES),
            "bbox": [x1, y1, x1 + w, y1 + h],
            "confidence": round(random.uniform(0.6, 1.0), 3),
        })
    return image, annotations

frame, annots = generate_synthetic_frame()
print(f"Frame shape: {list(frame.shape)}")
print(f"Annotations ({len(annots)}):")
for a in annots:
    print(f"  {a['class']:20s} bbox={a['bbox']}  conf={a['confidence']}")

In [ ]:
# ── COCO-format Export Preview ────────────────────────────────

def to_coco_format(frames_with_annots):
    """Converts list of (image_meta, annotations) to COCO JSON structure."""
    images, annotations = [], []
    ann_id = 1
    categories = {name: idx for idx, name in enumerate(SIGN_CLASSES)}

    for img_id, (_, annots) in enumerate(frames_with_annots, 1):
        images.append({"id": img_id, "width": 256, "height": 256, "file_name": f"frame_{img_id:05d}.png"})
        for a in annots:
            x1, y1, x2, y2 = a["bbox"]
            annotations.append({
                "id": ann_id, "image_id": img_id,
                "category_id": categories[a["class"]],
                "bbox": [x1, y1, x2-x1, y2-y1],    # COCO uses xywh
                "area": (x2-x1) * (y2-y1),
                "iscrowd": 0,
            })
            ann_id += 1

    return {
        "images": images,
        "annotations": annotations,
        "categories": [{"id": v, "name": k} for k, v in categories.items()],
    }

batch = [generate_synthetic_frame() for _ in range(10)]
coco = to_coco_format(batch)
print(f"COCO export: {len(coco['images'])} images, {len(coco['annotations'])} annotations, {len(coco['categories'])} categories")
print("\nSample annotation:")
print(json.dumps(coco["annotations"][0], indent=2))

---
## 4 · Experiment Tracking

Demonstrates the experiment manager that logs training runs with metrics, hyperparameters, and results.

In [ ]:
# ── Experiment Manager (in-memory demo) ──────────────────────
from datetime import datetime

class ExperimentManager:
    def __init__(self):
        self.experiments = []

    def create(self, name, config, model_type="student"):
        exp = {
            "id": len(self.experiments) + 1,
            "name": name,
            "model_type": model_type,
            "config": config,
            "status": "running",
            "created": datetime.now().isoformat(),
            "results": [],
            "metrics": {},
        }
        self.experiments.append(exp)
        return exp

    def log_epoch(self, exp_id, epoch, loss, accuracy, val_loss, val_accuracy):
        exp = self.experiments[exp_id - 1]
        exp["results"].append({
            "epoch": epoch, "loss": loss, "accuracy": accuracy,
            "val_loss": val_loss, "val_accuracy": val_accuracy,
        })

    def finish(self, exp_id, final_metrics):
        exp = self.experiments[exp_id - 1]
        exp["status"] = "completed"
        exp["metrics"] = final_metrics
        exp["finished"] = datetime.now().isoformat()

    def summary(self, exp_id):
        exp = self.experiments[exp_id - 1]
        return {
            "name": exp["name"],
            "status": exp["status"],
            "epochs": len(exp["results"]),
            "best_val_acc": max((r["val_accuracy"] for r in exp["results"]), default=0),
            "final_metrics": exp["metrics"],
        }

# Log our earlier distillation run
em = ExperimentManager()
exp = em.create("KD-ResNet50-to-Student-v1", {
    "teacher": "resnet50", "student": "custom_cnn",
    "temperature": 4.0, "alpha": 0.7, "beta": 0.15,
    "lr": LR, "epochs": EPOCHS, "batch_size": BATCH,
})

for i, epoch in enumerate(history["epoch"]):
    # We don't have separate train accuracy in our loop, use val as proxy
    em.log_epoch(exp["id"], epoch,
                 history["train_loss"][i], history["val_acc"][i],
                 history["val_loss"][i], history["val_acc"][i])

em.finish(exp["id"], {
    "final_val_accuracy": history["val_acc"][-1],
    "final_val_loss": history["val_loss"][-1],
    "student_params": sum(p.numel() for p in student.parameters()),
    "teacher_params": sum(p.numel() for p in teacher.parameters()),
    "compression_ratio": round(sum(p.numel() for p in teacher.parameters()) / sum(p.numel() for p in student.parameters()), 1),
})

summary = em.summary(exp["id"])
print("Experiment Summary:")
print(json.dumps(summary, indent=2))

---
## 5 · Model Inference

Demonstrates loading a model and running batch predictions.

In [ ]:
# ── Inference Service ─────────────────────────────────────────

class InferenceService:
    def __init__(self, model, device=DEVICE):
        self.model = model.to(device).eval()
        self.device = device
        self.class_names = ["Lane Keep", "Turn Left", "Turn Right", "Slow Down", "Stop"]

    @torch.no_grad()
    def predict(self, x):
        if isinstance(x, np.ndarray):
            x = torch.from_numpy(x).float()
        if x.dim() == 3:
            x = x.unsqueeze(0)
        x = x.to(self.device)
        logits = self.model(x)
        if isinstance(logits, tuple):
            logits = logits[0]
        probs = torch.softmax(logits, dim=1)
        pred_class = probs.argmax(1)
        return {
            "predictions": [self.class_names[c] for c in pred_class.cpu().tolist()],
            "confidences": probs.max(1).values.cpu().tolist(),
            "all_probs":   probs.cpu().tolist(),
        }

# Run inference with the trained student model
svc = InferenceService(student)

test_batch = torch.randn(8, 3, IMG_SIZE, IMG_SIZE)
results = svc.predict(test_batch)

print("Batch Predictions:")
for i, (pred, conf) in enumerate(zip(results["predictions"], results["confidences"])):
    print(f"  Sample {i+1}: {pred:12s}  confidence={conf:.3f}")

In [ ]:
# ── Quantum Inference ─────────────────────────────────────────
q_svc = InferenceService(hybrid_model)

test_features = torch.randn(8, INPUT_DIM)
q_results = q_svc.predict(test_features)

print("Quantum Hybrid Predictions:")
for i, (pred, conf) in enumerate(zip(q_results["predictions"], q_results["confidences"])):
    print(f"  Sample {i+1}: {pred:12s}  confidence={conf:.3f}")

---
## 6 · Model Analysis & Comparison

Compare parameter counts, FLOPs, and inference speed between models.

In [ ]:
# ── Model Comparison Table ────────────────────────────────────

def count_params(model):
    return sum(p.numel() for p in model.parameters())

def benchmark_inference(model, sample_input, n_runs=50):
    model.eval()
    sample_input = sample_input.to(DEVICE)
    # Warm-up
    with torch.no_grad():
        for _ in range(5):
            model(sample_input)
    # Timed runs
    start = time.time()
    with torch.no_grad():
        for _ in range(n_runs):
            model(sample_input)
    elapsed = (time.time() - start) / n_runs * 1000  # ms
    return elapsed

models_info = {
    "Teacher (ResNet-50)": {
        "model": teacher,
        "sample": torch.randn(1, 3, IMG_SIZE, IMG_SIZE),
    },
    "Student (Custom CNN)": {
        "model": student,
        "sample": torch.randn(1, 3, IMG_SIZE, IMG_SIZE),
    },
    "Quantum Hybrid": {
        "model": hybrid_model,
        "sample": torch.randn(1, INPUT_DIM),
    },
}

print(f"{'Model':<25} {'Parameters':>12} {'Inference (ms)':>15}")
print("─" * 55)
for name, info in models_info.items():
    params = count_params(info["model"])
    latency = benchmark_inference(info["model"], info["sample"])
    print(f"{name:<25} {params:>12,} {latency:>14.2f}")

compression = count_params(teacher) / count_params(student)
print(f"\nCompression ratio (Teacher/Student): {compression:.1f}×")

In [ ]:
# ── Confusion Matrix Visualization ────────────────────────────
from sklearn.metrics import confusion_matrix, classification_report

# Generate predictions for confusion matrix
student.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        logits, _ = student(images)
        all_preds.extend(logits.argmax(1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

class_names = ["Lane Keep", "Turn Left", "Turn Right", "Slow Down", "Stop"]
cm = confusion_matrix(all_labels, all_preds, labels=range(NUM_CLASSES))

fig, ax = plt.subplots(figsize=(8, 6))
fig.patch.set_facecolor("#0B0F19")
ax.set_facecolor("#121826")

im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
ax.set_title("Student Model – Confusion Matrix", color="white", fontsize=14)
ax.set_xlabel("Predicted", color="#8A94A6")
ax.set_ylabel("True", color="#8A94A6")
ax.set_xticks(range(NUM_CLASSES))
ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(class_names, rotation=45, ha="right", color="#8A94A6")
ax.set_yticklabels(class_names, color="#8A94A6")

for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "#5D8CFF", fontsize=12)

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))

---
## 7 · Save & Export

Save trained models for deployment in the Django application.

In [ ]:
# ── Save Checkpoints ─────────────────────────────────────────
os.makedirs("checkpoints", exist_ok=True)

# Student model
student_ckpt = {
    "model_state_dict": student.state_dict(),
    "num_classes": NUM_CLASSES,
    "img_size": IMG_SIZE,
    "val_accuracy": history["val_acc"][-1],
    "epochs_trained": EPOCHS,
    "config": {"lr": LR, "batch_size": BATCH, "temperature": 4.0, "alpha": 0.7},
}
torch.save(student_ckpt, "checkpoints/student_distilled.pth")

# Quantum hybrid model
quantum_ckpt = {
    "model_state_dict": hybrid_model.state_dict(),
    "num_classes": NUM_CLASSES,
    "input_dim": INPUT_DIM,
    "n_qubits": N_QUBITS,
    "n_layers": N_LAYERS,
    "epochs_trained": QN_EPOCHS,
}
torch.save(quantum_ckpt, "checkpoints/quantum_hybrid.pth")

print("Saved checkpoints:")
for f in os.listdir("checkpoints"):
    size = os.path.getsize(f"checkpoints/{f}") / 1024
    print(f"  {f:30s} {size:>8.1f} KB")

In [ ]:
# ── Final Summary ────────────────────────────────────────────

print("="*60)
print("  QuantumDrive Research Notebook – Summary")
print("="*60)
print(f"")
print(f"  Knowledge Distillation:")
print(f"    Teacher params      : {count_params(teacher):>12,}")
print(f"    Student params      : {count_params(student):>12,}")
print(f"    Compression ratio   : {count_params(teacher)/count_params(student):.1f}×")
print(f"    Best val accuracy   : {max(history['val_acc']):.1f}%")
print(f"")
print(f"  Quantum Hybrid:")
print(f"    Hybrid params       : {count_params(hybrid_model):>12,}")
print(f"    Best train accuracy : {max(h['acc'] for h in q_history):.1f}%")
print(f"    Qubits / Layers     : {N_QUBITS} / {N_LAYERS}")
print(f"")
print(f"  Synthetic Data:")
print(f"    COCO annotations    : {len(coco['annotations'])} across {len(coco['images'])} frames")
print(f"    Object categories   : {len(coco['categories'])}")
print(f"")
print(f"  Checkpoints saved to  : ./checkpoints/")
print("="*60)

---

**Next Steps:**
- Replace synthetic data with real CARLA-generated driving frames
- Deploy models to the Django backend via `inference/inference_service.py`  
- Track experiments in the Research Lab dashboard at `/research/`
- Scale quantum layers to more qubits as hardware improves

---

*Built with QuantumDrive — Django · PyTorch · PennyLane · Chart.js*